In [0]:
%python

import requests
import os 
import json
from dotenv import load_dotenv
from datetime import datetime, timezone

load_dotenv()

api_key = os.getenv('FH_API_KEY')
symbol = 'APPL'
ingested_at = datetime.now(timezone.utc).isoformat()


BRONZE_TABLE = "bronze.finnhub_company_news" 
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze LOCATION '/mnt/bronze/'")

base_url = 'https://finnhub.io/api/v1'

news_url = f'{base_url}/company-news'

news_params = {
    "symbol": symbol,
    "from": "2026-01-01",
    "to": "2026-06-20",
    "token": api_key,
}

news_response = requests.get(news_url, params=news_params)

news_bronze_record = {
    "source": "finnhub",
    "endpoint": "company-news",
    "symbol": symbol,
    "ingested_at": ingested_at,
    "status_code": news_response.status_code,
    "raw_payload": news_response.json() if news_response.status_code == 200 else None,
}

df = spark.createDataFrame([news_bronze_record])
 
df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
 
print(f"Status code: {news_bronze_record['status_code']}")
display(df)